# ⚔️ 03. RGB Training SVM & Injeksi Final (.keras)

Notebook ini melatih Support Vector Machine (SVM) dengan RBF Kernel pada vektor fitur CNN hasil ekstraksi Tahap 2.
- **Anti Data Leakage**: StandardScaler hanya di-it pada data training.
- **Injeksi Final .keras**: Model dasar CNN, StandardScaler, dan model SVM Classifier disimpan bersama ke dalam **SATU file tunggal** custom_cnn_cifar10_final.keras via format HDF5.


In [1]:
import os
import sys
import h5py
import pickle
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ARTIFACT_DIR = 'cifar10_artifacts/rgb'
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
print(f"Memuat vektor fitur training dari: {train_feat_path}")
train_data = np.load(train_feat_path)
X_train_features = train_data['X_train_features']
y_train = train_data['y_train']

RANDOM_STATE = 23092026

def save_and_inject_final_model(cnn_model, scaler, svm_model, artifact_path, metadata=None):
    os.makedirs(os.path.dirname(artifact_path), exist_ok=True)
    cnn_model.save(artifact_path)
    scaler_bytes = pickle.dumps(scaler, protocol=pickle.HIGHEST_PROTOCOL)
    svm_bytes = pickle.dumps(svm_model, protocol=pickle.HIGHEST_PROTOCOL)
    with h5py.File(artifact_path, 'a') as h5f:
        if 'scaler_pickle' in h5f:
            del h5f['scaler_pickle']
        if 'svm_model_pickle' in h5f:
            del h5f['svm_model_pickle']
        if 'svm_pipeline_pickle' in h5f:
            del h5f['svm_pipeline_pickle']
        h5f.create_dataset('scaler_pickle', data=np.frombuffer(scaler_bytes, dtype=np.uint8), compression='gzip')
        h5f.create_dataset('svm_model_pickle', data=np.frombuffer(svm_bytes, dtype=np.uint8), compression='gzip')
        if metadata:
            for k, v in metadata.items():
                if isinstance(v, (int, float, str)):
                    h5f.attrs[k] = v
                else:
                    h5f.attrs[k] = str(v)
    print(f"[BERHASIL] Seluruh model (CNN + Scaler + SVM) tersimpan dalam SATU file (.keras): {artifact_path}")

def load_injected_final_model(artifact_path):
    if not os.path.exists(artifact_path):
        raise FileNotFoundError(f"File model {artifact_path} tidak ditemukan!")
    cnn_model = tf.keras.models.load_model(artifact_path)
    with h5py.File(artifact_path, 'r') as h5f:
        if 'scaler_pickle' in h5f and 'svm_model_pickle' in h5f:
            scaler_bytes = bytes(h5f['scaler_pickle'][()])
            svm_bytes = bytes(h5f['svm_model_pickle'][()])
            scaler = pickle.loads(scaler_bytes)
            svm_model = pickle.loads(svm_bytes)
        elif 'svm_pipeline_pickle' in h5f:
            pipe_bytes = bytes(h5f['svm_pipeline_pickle'][()])
            pipe_obj = pickle.loads(pipe_bytes)
            if hasattr(pipe_obj, 'named_steps'):
                scaler = pipe_obj.named_steps.get('scaler', None)
                svm_model = pipe_obj.named_steps.get('svm', pipe_obj)
            else:
                scaler = None
                svm_model = pipe_obj
        else:
            raise KeyError(f"Dataset Scaler/SVM tidak ditemukan dalam {artifact_path}!")
        metadata = {}
        for k, v in h5f.attrs.items():
            if isinstance(v, bytes):
                metadata[k] = v.decode('utf-8')
            else:
                metadata[k] = v
    return cnn_model, scaler, svm_model, metadata


Memuat vektor fitur training dari: cifar10_artifacts/rgb\train_features.npz


## 🛠️ 1. Standarisasi Fitur & Melatih Model SVM Classifier (Anti Data Leakage)


In [2]:
# 1. Standarisasi Fitur Training (Fit & Transform pada data training saja - Anti Data Leakage)
scaler = StandardScaler()
print("Menstandarisasi vektor fitur training dengan StandardScaler...")
X_train_scaled = scaler.fit_transform(X_train_features)
print(f"[BERHASIL] Fitur training telah distandarisasi. Shape: {X_train_scaled.shape}")

print()
print("2. Melatih Support Vector Machine (RBF Kernel, C=10.0, gamma='scale')...")
svm_model = SVC(
    C=10.0,
    gamma='scale',
    kernel='rbf',
    cache_size=4096,
    random_state=RANDOM_STATE,
    decision_function_shape='ovr'
)
svm_model.fit(X_train_scaled, y_train)
print("[BERHASIL] SVM Classifier telah selesai dilatih!")

sample_acc = accuracy_score(y_train[:5000], svm_model.predict(X_train_scaled[:5000]))
print(f"Akurasi SVM pada sampel training: {sample_acc * 100:.2f}%")

Menstandarisasi vektor fitur training dengan StandardScaler...
[BERHASIL] Fitur training telah distandarisasi. Shape: (50000, 512)

2. Melatih Support Vector Machine (RBF Kernel, C=10.0, gamma='scale')...
[BERHASIL] SVM Classifier telah selesai dilatih!
Akurasi SVM pada sampel training: 98.86%


## 📦 2. Menyimpan Scaler & Model SVM ke File Model .keras Tunggal


In [3]:
cnn_model_path = os.path.join(ARTIFACT_DIR, 'cnn_model.keras')
cnn_model = tf.keras.models.load_model(cnn_model_path)
final_keras_path = os.path.join(ARTIFACT_DIR, 'custom_cnn_cifar10_final.keras')

metadata = {
    'method': 'RGB CNN + SVM',
    'feature': 'RGB',
    'svm_C': 10.0,
    'svm_gamma': 'scale',
    'feature_dim': int(X_train_features.shape[1])
}

save_and_inject_final_model(cnn_model, scaler, svm_model, final_keras_path, metadata)

v_cnn, v_scaler, v_svm, v_meta = load_injected_final_model(final_keras_path)
print(f"[VERIFIKASI] Model final .keras berhasil memuat CNN ({v_cnn.name}), Scaler ({v_scaler}), dan SVM ({v_svm})!")

[BERHASIL] Seluruh model (CNN + Scaler + SVM) tersimpan dalam SATU file (.keras): cifar10_artifacts/rgb\custom_cnn_cifar10_final.keras
[VERIFIKASI] Model final .keras berhasil memuat CNN (cnn_3ch), Scaler (StandardScaler()), dan SVM (SVC(C=10.0, cache_size=4096, random_state=23092026))!
